#  UMBC DATA606 — Flight Delay Prediction: Exploratory Data Analysis (Regression Focus)

### Objective

This notebook performs a **comprehensive exploratory data analysis (EDA)** on U.S. flight performance data to support a **regression-based prediction** system for flight delays.

We aim to:
1. **Predict the departure delay** (in minutes).  
2. **Predict the arrival delay** (in minutes).

---

### Context

According to the **U.S. Department of Transportation (DOT)**, a flight is considered **delayed** if it departs or arrives **more than 15 minutes** after its scheduled time.  
In this project, the 15-minute threshold is used only for *interpretation*, not for modeling — our models will predict continuous delay values (in minutes).

---

###  EDA Goals

- Assess dataset quality (shape, missingness, KPIs).  
- Understand the distributions and skew of `DepDelayMinutes` and `ArrDelayMinutes`.  
- Identify temporal, airline, and airport-level patterns affecting delays.  
- Detect correlations and outliers to guide feature selection.  
- Define data capping and feature readiness for the regression models.

---

### Key Outputs
- Clean and analyzed dataset ready for modeling.  
- Clear feature insights and delay behavior visualizations.  
- Final list of predictive variables to be used in modeling notebook.

In [1]:
!pip install plotly
!pip install plotly[express]
!pip install --upgrade nbformat

zsh:1: no matches found: plotly[express]


In [2]:
!pyspark --version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/06 11:14:50 WARN Utils: Your hostname, Drashis-MacBook-Air-8.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.229 instead (on interface en0)
25/11/06 11:14:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.0.1
      /_/
                        
Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 21.0.8
Branch HEAD
Compiled by user runner on 2025-09-02T03:10:51Z
Revision 29434ea766b0fc3c3bf6eaadb43a8f931133649e
Url https://github.com/apache/spark
Type --help for more information.


In [3]:
from pyspark.sql import SparkSession
import plotly.express as px

In [4]:
spark = SparkSession.builder \
    .appName("FlightDelay") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/06 11:14:51 WARN Utils: Your hostname, Drashis-MacBook-Air-8.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.229 instead (on interface en0)
25/11/06 11:14:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/06 11:14:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/06 11:14:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
df = spark.read.csv( "/Users/drashi/Documents/UMBC-DATA606-Capstone/Data/combined_flight_data.csv", header=True, inferSchema=True)

### PHASE 1 — DATA CLEANING & SCHEMA SETUP

#### Basic Setup & Schema Inspection

In [6]:
from pyspark.sql import functions as F

# Show record count and schema
print(f"Total records: {df.count():,}")
df.printSchema()

Total records: 17,373,636
root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- DOT_ID_Reporting_Airline: integer (nullable = true)
 |-- IATA_CODE_Reporting_Airline: string (nullable = true)
 |-- Tail_Number: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: double (nullable = true)
 |-- OriginAirportID: integer (nullable = true)
 |-- OriginAirportSeqID: integer (nullable = true)
 |-- OriginCityMarketID: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- OriginStateFips: integer (nullable = true)
 |-- OriginStateName: string (nullable = true)
 |-- OriginWac: integer (nullable = true)
 |-- DestAirportID: integer (nullable = true

In [7]:
# Quick look at data
df.show(5, truncate=False)

25/11/06 11:15:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+--------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+----------------+--------+--------------+-----------------+-------+-------+--------+-------------+------------+------------+--------+-------------+-----------------+------------+-------------+---------------+------------------+--------------+--------------------+-----------+-----------+-----------+-------------+----------------+------------+--------------+----------------+----

#### Filter Out Cancelled & Diverted Flights

Cancelled or diverted flights can distort delay calculations. We will have only valid, completed flights.

In [8]:
df = df.filter((F.col("Cancelled") == 0) & (F.col("Diverted") == 0))

#### Drop Irrelevant or Redundant Columns

In [10]:
#### Drop Irrelevant or Redundant Columns

cols_to_drop = [
    'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number',
    'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID',
    'OriginStateFips', 'OriginStateName', 'OriginWac',
    'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID',
    'DestStateFips', 'DestStateName', 'DestWac',
    'Cancelled', 'CancellationCode', 'Diverted',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
    'FirstDepTime', 'TotalAddGTime', 'LongestAddGTime',
    'DivAirportLandings', 'DivReachedDest', 'DivActualElapsedTime', 'DivArrDelay', 'DivDistance'
]

df = df.drop(*cols_to_drop)

#### Count Null (and Missing) Values per Column

In [11]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# numeric and string columns
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

# For string columns: check both NULL and empty string
# For numeric columns: check only NULL
null_counts_expr = [
    F.count(
        F.when(
            F.col(c).isNull() | ((F.col(c) == '') if c in string_cols else F.lit(False)),
            c
        )
    ).alias(c)
    for c in df.columns
]

null_counts = df.select(null_counts_expr)
null_counts.show(truncate=False, vertical=True)


-RECORD 0-----------------------------------
 Year                            | 0        
 Quarter                         | 0        
 Month                           | 0        
 DayofMonth                      | 0        
 DayOfWeek                       | 0        
 FlightDate                      | 0        
 Reporting_Airline               | 0        
 Flight_Number_Reporting_Airline | 1        
 Origin                          | 0        
 OriginCityName                  | 0        
 OriginState                     | 0        
 Dest                            | 0        
 DestCityName                    | 0        
 DestState                       | 0        
 CRSDepTime                      | 0        
 DepTime                         | 0        
 DepDelay                        | 0        
 DepDelayMinutes                 | 0        
 DepDel15                        | 0        
 DepartureDelayGroups            | 0        
 DepTimeBlk                      | 0        
 TaxiOut  

Dropping the Div* Columns as these columns (Div1Airport, Div2Airport, Div3AirportID, etc.) store information only for flights that were diverted and they didn’t land at their scheduled destination.

In [12]:
# Drop all Div columns (Div1Airport, Div2Airport, etc.)
div_cols = [c for c in df.columns if c.startswith("Div")]
df = df.drop(*div_cols)

#### Handle Missing Values 

Our goals here are to:
1. Critical fields (like Reporting_Airline, Origin, Dest, FlightDate): drop rows if null.
2. Numeric time/delay fields (DepDelay, ArrDelay, TaxiOut, TaxiIn, etc.): fill with 0 (because 0 = “no delay” or “not applicable”).
3. Extra or empty string columns: drop or clean. -

In [13]:
# 1️. Drop rows missing critical info
critical_cols = ['Reporting_Airline', 'Origin', 'Dest', 'FlightDate']
df = df.dropna(subset=critical_cols)

In [14]:
# 2️. Replace nulls in numeric columns with 0
numeric_cols = [
    'DepDelay', 'DepDelayMinutes', 'ArrDelay', 'ArrDelayMinutes',
    'TaxiOut', 'TaxiIn', 'CRSElapsedTime', 'ActualElapsedTime',
    'AirTime', 'Distance', 'DistanceGroup', 'Flights'
]
df = df.fillna(0, subset=[c for c in numeric_cols if c in df.columns])

In [15]:
# 3️. Replace nulls in binary/delay indicators with 0
binary_cols = ['DepDel15', 'ArrDel15']
df = df.fillna(0, subset=[c for c in binary_cols if c in df.columns])

In [16]:
# 4️. Drop any unnamed or completely empty columns
df = df.drop('Unnamed: 109') if 'Unnamed: 109' in df.columns else df

In [17]:
# recheck for remaining nulls
null_counts = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show(truncate=False, vertical=True)

-RECORD 0------------------------------
 Year                            | 0   
 Quarter                         | 0   
 Month                           | 0   
 DayofMonth                      | 0   
 DayOfWeek                       | 0   
 FlightDate                      | 0   
 Reporting_Airline               | 0   
 Flight_Number_Reporting_Airline | 1   
 Origin                          | 0   
 OriginCityName                  | 0   
 OriginState                     | 0   
 Dest                            | 0   
 DestCityName                    | 0   
 DestState                       | 0   
 CRSDepTime                      | 0   
 DepTime                         | 0   
 DepDelay                        | 0   
 DepDelayMinutes                 | 0   
 DepDel15                        | 0   
 DepartureDelayGroups            | 0   
 DepTimeBlk                      | 0   
 TaxiOut                         | 0   
 WheelsOff                       | 0   
 WheelsOn                        | 1   


In [18]:
df.select('DepDelayMinutes', 'ArrDelayMinutes').summary().show()
print(f" Final record count after cleaning: {df.count():,}")

+-------+------------------+------------------+
|summary|   DepDelayMinutes|   ArrDelayMinutes|
+-------+------------------+------------------+
|  count|          17094473|          17094473|
|   mean|15.759770658036665|15.827009115753379|
| stddev| 54.56062817201303|54.497502380948745|
|    min|               0.0|               0.0|
|    25%|               0.0|               0.0|
|    50%|               0.0|               0.0|
|    75%|               9.0|              10.0|
|    max|            4413.0|            4405.0|
+-------+------------------+------------------+



 Final record count after cleaning: 17,094,473


### PHASE 2 — EXPLORATORY DATA ANALYSIS

#### Distribution of Departure and Arrival Delays

Goal: Understand how departure and arrival delays are distributed — identify whether most flights are on time, early, or significantly delayed.

In [19]:
delayed_df = df.filter(F.col("DepDelayMinutes") > 15).filter(F.col("ArrDelayMinutes") > 15)

In [20]:
from pyspark.sql import functions as F
import plotly.express as px

#  delay bins (every 5 minutes)
# Floor division groups delays like 0-4, 5-9, 10-14, etc.
dep_delay_hist = (
    delayed_df.withColumn("DepDelayBin", (F.floor(F.col("DepDelayMinutes") / 5) * 5))
      .groupBy("DepDelayBin")
      .agg(F.count("*").alias("FlightCount"))
      .orderBy("DepDelayBin")
)

arr_delay_hist = (
    delayed_df.withColumn("ArrDelayBin", (F.floor(F.col("ArrDelayMinutes") / 5) * 5))
  
      .groupBy("ArrDelayBin")
      .agg(F.count("*").alias("FlightCount"))
      .orderBy("ArrDelayBin")
)

In [21]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Convert PySpark grouped results to Pandas
dep_pdf = dep_delay_hist.toPandas()
arr_pdf = arr_delay_hist.toPandas()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Departure Delays (>15 min)", "Arrival Delays (>15 min)")
)

fig.add_trace(
    go.Box(
        y=dep_pdf["DepDelayBin"],
        name="Departure Delay",
        boxpoints="outliers"  # show outliers
    ),
    row=1, col=1
)

# Arrival Delay box plot
fig.add_trace(
    go.Box(
        y=arr_pdf["ArrDelayBin"],
        name="Arrival Delay",
        boxpoints="outliers"
    ),
    row=1, col=2
)


fig.update_layout(
    title_text="Comparison of Departure vs Arrival Delays (5-Minute Bins)",
    showlegend=False,
    height=500,
    width=1000
)

# Focus y-axis range
# fig.update_yaxes(range=[0, 500], title_text="Delay (minutes)", row=1, col=1)

# fig.update_yaxes(range=[0, 500], title_text="Delay (minutes)", row=1, col=2)

fig.show()


- The boxplots show that both departure and arrival delays have a similar distribution, with most delays clustering below 2000 minutes but a few extreme outliers extending beyond 4000 minutes. 
- Overall, arrival delays tend to slightly mirror departure delays, suggesting that late departures often lead to late arrivals.

#### Average Delay by Month

Goal: Identify which months experience the highest and lowest average departure and arrival delays to uncover seasonal trends in flight performance.

In [22]:
from pyspark.sql import functions as F
import plotly.express as px


avg_delay_by_month = (
    delayed_df.groupBy("Month")
      .agg(
          F.avg("DepDelayMinutes").alias("AvgDepDelay"),
          F.avg("ArrDelayMinutes").alias("AvgArrDelay"),
          F.count("*").alias("FlightCount")
      )
      .orderBy("Month")
)

# Convert to Pandas
avg_delay_month_pdf = avg_delay_by_month.toPandas()


fig = px.line(
    avg_delay_month_pdf,
    x="Month",
    y=["AvgDepDelay", "AvgArrDelay"],
    markers=True,
    title="Average Departure and Arrival Delay by Month",
    labels={"value": "Average Delay (minutes)", "Month": "Month", "variable": "Delay Type"}
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

Interpretation:
- Peak delays occur in June and July, likely due to summer travel congestion and thunderstorms.
- January also shows elevated delays, reflecting winter weather disruptions.
- Lowest delays appear around October–November, indicating smoother seasonal operations.
- Arrival delays consistently track slightly below departure delays, suggesting that flights often make up some lost time en route.

Overall Insight:
- Flight delays show clear seasonal patterns: highest in summer and winter, lowest in fall,  highlighting the strong influence of weather and travel demand on delay behavior.

#### Average Delay by Day of Week

Goal: Identify which days of the week experience higher or lower flight delays — useful for detecting operational or demand-based trends (e.g., business travel vs leisure peaks).

In [32]:
from pyspark.sql import functions as F
import plotly.express as px

# --- Group & aggregate ---
avg_delay_by_dow = (
    delayed_df.groupBy("DayOfWeek")
      .agg(
          F.avg("DepDelayMinutes").alias("AvgDepDelay"),
          F.avg("ArrDelayMinutes").alias("AvgArrDelay"),
          F.count("*").alias("FlightCount")
      )
)

# --- Convert to pandas & compute total average ---
avg_delay_dow_pdf = avg_delay_by_dow.toPandas()
avg_delay_dow_pdf["TotalAvgDelay"] = (
    avg_delay_dow_pdf["AvgDepDelay"] + avg_delay_dow_pdf["AvgArrDelay"]
) / 2

# --- Sort descending by total average delay ---
avg_delay_dow_pdf = avg_delay_dow_pdf.sort_values("TotalAvgDelay", ascending=False)

# --- Create grouped bar chart ---
fig = px.bar(
    avg_delay_dow_pdf,
    x="DayOfWeek",
    y=["AvgDepDelay", "AvgArrDelay"],
    barmode="group",
    title="Average Departure and Arrival Delay by Day of Week (Descending Order)",
    labels={
        "value": "Average Delay (minutes)",
        "DayOfWeek": "Day of Week",
        "variable": "Delay Type"
    }
)

# --- Optional: update tick labels for weekdays ---
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=avg_delay_dow_pdf["DayOfWeek"],
        ticktext=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    )
)

fig.show()


- The chart shows that average departure and arrival delays remain fairly consistent across all days of the week, with only slight variations. 
- This suggests that day of the week has minimal influence on flight delays, indicating delays are likely driven by other operational or external factors.

#### Average Delay by Hour of Day
Goal: Understand how flight delays vary throughout the day — from early morning departures to late-night flights — to detect daily congestion or cascading delay effects.

In [24]:
from pyspark.sql import functions as F
import plotly.express as px


delayed_df = delayed_df.withColumn("DepHour", (F.col("CRSDepTime") / 100).cast("int"))

# Compute average departure and arrival delays per hour
avg_delay_by_hour = (
    delayed_df.groupBy("DepHour")
      .agg(
          F.avg("DepDelay").alias("AvgDepDelay"),
          F.avg("ArrDelay").alias("AvgArrDelay"),
          F.count("*").alias("FlightCount")
      )
      .orderBy("DepHour")
)

# Convert to Pandas for plotting
avg_delay_hour_pdf = avg_delay_by_hour.toPandas()

# Plot 
fig = px.line(
    avg_delay_hour_pdf,
    x="DepHour",
    y=["AvgDepDelay", "AvgArrDelay"],
    markers=True,
    title="Average Departure and Arrival Delay by Hour of Day",
    labels={
        "DepHour": "Scheduled Departure Hour (Local Time)",
        "value": "Average Delay (minutes)",
        "variable": "Delay Type"
    }
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

- The chart shows that flight delays peak sharply between 5–6 AM, likely due to early operational congestion or scheduling overlaps. 
- After this spike, delays remain relatively stable throughout the day, with only a slight rise again late at night.

#### Average Delay by Airline 
Goal: Compare average departure and arrival delays across different airlines to assess carrier-level performance and reliability.

In [25]:
delayed_df.select("Reporting_Airline").distinct().orderBy("Reporting_Airline").show(truncate=False)

+-----------------+
|Reporting_Airline|
+-----------------+
|9E               |
|AA               |
|AS               |
|B6               |
|DL               |
|F9               |
|G4               |
|HA               |
|MQ               |
|NK               |
|OH               |
|OO               |
|UA               |
|WN               |
|YX               |
+-----------------+



In [33]:
from pyspark.sql import functions as F
import plotly.express as px

# --- Aggregate average delays by airline ---
avg_delay_by_airline = (
    delayed_df.groupBy("Reporting_Airline")
      .agg(
          F.avg("DepDelay").alias("AvgDepDelay"),
          F.avg("ArrDelay").alias("AvgArrDelay"),
          F.count("*").alias("FlightCount")
      )
)

# --- Convert to pandas ---
avg_delay_airline_pdf = avg_delay_by_airline.toPandas()

# --- Airline code-to-name mapping ---
airline_name_map = {
    "9E": "Endeavor Air",
    "AA": "American Airlines",
    "AS": "Alaska Airlines",
    "B6": "JetBlue Airways",
    "DL": "Delta Air Lines",
    "F9": "Frontier Airlines",
    "G4": "Allegiant Air",
    "HA": "Hawaiian Airlines",
    "MQ": "Envoy Air",
    "NK": "Spirit Airlines",
    "OH": "PSA Airlines",
    "OO": "SkyWest Airlines",
    "UA": "United Airlines",
    "WN": "Southwest Airlines",
    "YX": "Republic Airways"
}

avg_delay_airline_pdf["AirlineName"] = avg_delay_airline_pdf["Reporting_Airline"].map(airline_name_map)

# --- Sort airlines by total average delay (descending) ---
avg_delay_airline_pdf["TotalAvgDelay"] = (
    avg_delay_airline_pdf["AvgDepDelay"] + avg_delay_airline_pdf["AvgArrDelay"]
) / 2

avg_delay_airline_pdf = avg_delay_airline_pdf.sort_values("TotalAvgDelay", ascending=False)

# --- Create grouped bar chart ---
fig = px.bar(
    avg_delay_airline_pdf,
    x="Reporting_Airline",
    y=["AvgDepDelay", "AvgArrDelay"],
    barmode="group",
    title="Average Departure and Arrival Delay by Airline (Descending Order)",
    labels={
        "Reporting_Airline": "Airline Code",
        "value": "Average Delay (minutes)",
        "variable": "Delay Type"
    },
    hover_data={
        "AirlineName": True,
        "FlightCount": True,
        "Reporting_Airline": False
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    bargap=0.2,
    legend_title_text="Delay Type",
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()


- The chart shows that average delays vary significantly across airlines, with carriers like AA (American), OH (PSA), and OO(SkyWest) showing higher delay times compared to others. 
- Airlines such as AS(Alaska Airlines) and WN(SouthWest) maintain relatively lower delays, suggesting more efficient scheduling or operations.

In [27]:
"""from pyspark.sql import functions as F
import plotly.express as px

# Compute average delays per destination airport
avg_delay_by_dest = (
    delayed_df.groupBy("Dest", "DestCityName")
      .agg(
          F.avg("DepDelay").alias("AvgDepDelay"),
          F.avg("ArrDelay").alias("AvgArrDelay"),
          F.count("*").alias("FlightCount")
      )
      .orderBy(F.desc("AvgArrDelay"))
      .limit(15)
)

# Convert to Pandas for plotting
avg_delay_dest_pdf = avg_delay_by_dest.toPandas()

# Create grouped bar chart
fig = px.bar(
    avg_delay_dest_pdf,
    x="Dest",
    y=["AvgDepDelay", "AvgArrDelay"],
    barmode="group",
    title="Top 15 Destination Airports by Average Departure and Arrival Delay",
    labels={
        "Dest": "Airport Code",
        "value": "Average Delay (minutes)",
        "variable": "Delay Type"
    },
    hover_data={
        "DestCityName": True,
        "FlightCount": True
    }
)

# Layout customization
fig.update_layout(
    xaxis_tickangle=-45,
    bargap=0.2,
    legend_title_text="Delay Type",
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()"""

'from pyspark.sql import functions as F\nimport plotly.express as px\n\n# Compute average delays per destination airport\navg_delay_by_dest = (\n    delayed_df.groupBy("Dest", "DestCityName")\n      .agg(\n          F.avg("DepDelay").alias("AvgDepDelay"),\n          F.avg("ArrDelay").alias("AvgArrDelay"),\n          F.count("*").alias("FlightCount")\n      )\n      .orderBy(F.desc("AvgArrDelay"))\n      .limit(15)\n)\n\n# Convert to Pandas for plotting\navg_delay_dest_pdf = avg_delay_by_dest.toPandas()\n\n# Create grouped bar chart\nfig = px.bar(\n    avg_delay_dest_pdf,\n    x="Dest",\n    y=["AvgDepDelay", "AvgArrDelay"],\n    barmode="group",\n    title="Top 15 Destination Airports by Average Departure and Arrival Delay",\n    labels={\n        "Dest": "Airport Code",\n        "value": "Average Delay (minutes)",\n        "variable": "Delay Type"\n    },\n    hover_data={\n        "DestCityName": True,\n        "FlightCount": True\n    }\n)\n\n# Layout customization\nfig.update_layou

#### Percentage of Delayed flight per top 10 Busiest Airports

Goal: A plot showing the top 10 busiest airports with the percentage of delayed flights

In [34]:
import pyspark.sql.functions as F
import plotly.express as px

# --- Aggregate airport delays ---
airport_delay_df = (
    df.groupBy("Origin", "OriginCityName")
      .agg(
          F.count("*").alias("TotalFlights"),
          F.sum("DepDel15").alias("DelayedFlights")
      )
      .withColumn("DelayPercent", (F.col("DelayedFlights") / F.col("TotalFlights")) * 100)
      .withColumn("WeightedScore", F.col("DelayPercent") * F.log(F.col("TotalFlights")))
      .orderBy(F.desc("WeightedScore"))
      .limit(10)
)

airport_delay_pdf = airport_delay_df.toPandas()

# --- Sort by DelayPercent (descending) for plotting ---
airport_delay_pdf = airport_delay_pdf.sort_values("DelayPercent", ascending=False)

# --- Create bar chart ---
fig = px.bar(
    airport_delay_pdf,
    x="Origin",
    y="DelayPercent",
    text=airport_delay_pdf["DelayPercent"].round(2).astype(str) + "%",
    title="Top 10 Airports by Percentage of Delayed Departures (Descending Order)",
    labels={"Origin": "Airport Code", "DelayPercent": "Percentage of Delayed Flights (%)"},
    hover_data={
        "Origin": True,
        "OriginCityName": True,
        "TotalFlights": True,
        "DelayedFlights": True,
        "DelayPercent": ":.2f"
    }
)

# --- Customize layout ---
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis=dict(title="Percentage of Delayed Flights (%)"),
    xaxis=dict(title="Airport Code"),
    showlegend=False,
    height=550
)

fig.show()

- Departure Delays Chart: ASE, BWI, and FLL experience the highest percentages of delayed departures, with ASE leading at over 31%, pointing to operational or congestion challenges. 
- Most major airports maintain delay percentages around 24%–28%, showing that departure delays are widespread but generally moderate.

In [ ]:
import pyspark.sql.functions as F
import plotly.express as px

# --- Aggregate arrival delays ---
arrival_delay_df = (
    df.groupBy("Dest", "DestCityName")
      .agg(
          F.count("*").alias("TotalFlights"),
          F.sum("ArrDel15").alias("DelayedFlights")
      )
      .withColumn("DelayPercent", (F.col("DelayedFlights") / F.col("TotalFlights")) * 100)
      .withColumn("WeightedScore", F.col("DelayPercent") * F.log(F.col("TotalFlights")))
      .orderBy(F.desc("WeightedScore"))
      .limit(10)
)

arrival_delay_pdf = arrival_delay_df.toPandas()

# --- Sort by DelayPercent descending for plotting ---
arrival_delay_pdf = arrival_delay_pdf.sort_values("DelayPercent", ascending=False)

# --- Create bar chart ---
fig = px.bar(
    arrival_delay_pdf,
    x="Dest",
    y="DelayPercent",
    text=arrival_delay_pdf["DelayPercent"].round(2).astype(str) + "%",
    title="Top 10 Airports by Percentage of Delayed Arrivals (Descending Order)",
    labels={"Dest": "Airport Code", "DelayPercent": "Percentage of Delayed Flights (%)"},
    hover_data={
        "Dest": True,
        "DestCityName": True,
        "TotalFlights": True,
        "DelayedFlights": True,
        "DelayPercent": ":.2f"
    }
)

# --- Formatting & layout ---
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis=dict(title="Percentage of Delayed Flights (%)"),
    xaxis=dict(title="Airport Code"),
    showlegend=False,
    height=550
)

fig.show()


25/11/07 03:01:04 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1068562 ms exceeds timeout 120000 ms
25/11/07 03:01:04 WARN SparkContext: Killing executors is not supported by current scheduler.
25/11/07 03:16:32 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

- Airports like ASE, SFB, and SJU show the highest percentages of delayed arrivals, with around 30% of flights delayed, indicating potential capacity or weather-related issues. 
- The overall range of arrival delays across top airports is between 23%–30%, suggesting relatively consistent delay patterns.

#### Correlation Analysis

Goal: To examine relationships between key numerical features — such as distance, airtime, departure delay, and arrival delay — in order to identify which variables are most influential for predicting delays.

In [30]:
from pyspark.sql import functions as F
import pandas as pd
import plotly.express as px


spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")


numeric_cols = [
    "DepDelay", "ArrDelay", "AirTime", "Distance",
    "TaxiOut", "TaxiIn", "CRSElapsedTime", "ActualElapsedTime"
]


sample_df = (
    delayed_df.select(numeric_cols)
      .sample(fraction=0.05, seed=42)
      .na.drop()
)

print("Converting 5% Spark sample to Pandas using Arrow...")
numeric_pdf = sample_df.toPandas()
print(f" Conversion done — shape: {numeric_pdf.shape}")


corr_df = numeric_pdf.corr()


fig = px.imshow(
    corr_df,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    title="Correlation Heatmap ",
    width=950,        
    height=800  
)
fig.update_layout(
    xaxis_title="Feature",
    yaxis_title="Feature",
    coloraxis_colorbar=dict(title="Correlation Coefficient"),
)
fig.show()

Converting 5% Spark sample to Pandas using Arrow...


 Conversion done — shape: (138245, 8)


This heatmap shows how different flight features relate to each other.
- Departure delay and arrival delay have a very strong correlation (0.99), meaning late departures almost always lead to late arrivals, while flight time and distance are also highly related.

In [31]:
# cleaned dataset for modeling 
out_path = "/Users/drashi/Documents/UMBC-DATA606-Capstone/Data/clean_flights.parquet"

df.write.mode("overwrite").parquet(out_path)

print("Saved cleaned dataset to Parquet:", out_path)
print("Rows:", df.count(), " | Cols:", len(df.columns))

25/11/06 11:18:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/11/06 11:18:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/11/06 11:19:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/11/06 11:19:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/11/06 11:19:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/11/06 11:19:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/11/06 11:19:08 WARN MemoryManager: Total allocation exceeds 95.00% 

Saved cleaned dataset to Parquet: /Users/drashi/Documents/UMBC-DATA606-Capstone/Data/clean_flights.parquet


Rows: 17094473  | Cols: 38


#### EDA Insights

- Delays are highly correlated: Departure delays almost always lead to arrival delays.
- Time patterns: Early morning (5–8 AM) flights show the highest delays, while mid-day flights experience the least.
- Day of week: Delay patterns remain fairly consistent across all days.
- Airlines: Some carriers (e.g., AA, OH, NK) tend to have higher delays than others.
- Airports: Specific airports like ASE, EKO, and OWB consistently show high delay percentages, possibly due to weather or operational constraints.
- Distance and AirTime: These variables are strongly correlated, showing that flight duration impacts delay length.

#### Overall Insights
- Flight delays are influenced by time of day, airline, and airport factors, with early morning and certain airports showing higher delays. 
- Departure and arrival delays are strongly correlated, and longer flights tend to experience greater delay variability.

#### Target Variable (Regression)

ArrDelay – Arrival delay in minutes (continuous value to be predicted).
DepDelay - Departure delay in minutes (continuous value to be predicted).

#### Key Features (Predictors)

1. Departure-related:
     CRSDepTime

2. Time & Schedule:
    - DayOfWeek, Month, CRSElapsedTime, ActualElapsedTime

3. Distance & AirTime:
    - Distance, AirTime

4. Operational & Geographic:
    - Origin, Dest, Carrier

These predictors capture timing, route, airline, and operational factors influencing arrival delay.